# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


####  Run this cell to set up and start your interactive session.


In [2]:
%stop_session

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
There is no current session.


In [5]:
%idle_timeout 30
%glue_version 6.0
%worker_type G.4X
%number_of_workers 10

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 6.0
Previous worker type: None
Setting new worker type to: G.4X
Previous number of workers: None
Setting new number of workers to: 10


In [8]:
%extra_jars s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar, s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar 
%extra_py_files s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip

Extra jars to be included:
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar
Extra py files to be included:
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar,s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar


In [35]:
#%%configure
#{
#  "--datalake-formats": "iceberg,delta",
#  "--additional-python-modules": "duckdb,pyarrow,shapely",
#  "--conf": {
#    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
#    "spark.kryo.registrator": "com.esri.geoanalytics.KryoRegistrator",
#    "spark.plugins": "com.esri.geoanalytics.Plugin",
#    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension",
#    "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog",
#    "spark.sql.catalog.glue_catalog": "org.apache.iceberg.spark.SparkCatalog",
#    "spark.sql.catalog.glue_catalog.catalog-impl": "org.apache.iceberg.aws.glue.GlueCatalog",
#    "spark.sql.catalog.glue_catalog.warehouse": "s3://pske-prd-customerexperienceadhoc/spatial_analysis/"
#  }}

The following configurations have been updated: {'--datalake-formats': 'iceberg,delta', '--additional-python-modules': 'duckdb,pyarrow,shapely', '--conf': {'spark.serializer': 'org.apache.spark.serializer.KryoSerializer', 'spark.kryo.registrator': 'com.esri.geoanalytics.KryoRegistrator', 'spark.plugins': 'com.esri.geoanalytics.Plugin', 'spark.sql.extensions': 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension', 'spark.sql.catalog.spark_catalog': 'org.apache.spark.sql.delta.catalog.DeltaCatalog', 'spark.sql.catalog.glue_catalog': 'org.apache.iceberg.spark.SparkCatalog', 'spark.sql.catalog.glue_catalog.catalog-impl': 'org.apache.iceberg.aws.glue.GlueCatalog', 'spark.sql.catalog.glue_catalog.warehouse': 's3://pske-prd-customerexperienceadhoc/spatial_analysis/'}}


In [10]:
%%configure
{
 "--conf": "spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin",
}

The following configurations have been updated: {'--conf': 'spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin'}


In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import expr
from awsglue import DynamicFrame
from pyspark.sql.functions import col, to_timestamp
import pyspark.sql.functions as F
import time

# Initialize Spark session
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
# spark = SparkSession.builder.getOrCreate()
spark = SparkSession.builder \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "com.esri.geoanalytics.KryoRegistrator") \
    .config("spark.plugins", "com.esri.geoanalytics.Plugin") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3://pske-prd-datalake/") \
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog") \
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .getOrCreate()
job = Job(glueContext)

# Fail loudly on a stuck broadcast instead of hanging silently (default is 300s).
spark.conf.set("spark.sql.broadcastTimeout", "1200")

# Check active session configs
print("Spark Extensions:", spark.conf.get("spark.sql.extensions", "None"))
print("Serializer:", spark.conf.get("spark.serializer", "None"))
print("Spark Session successfully instantiated!")


Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.4X
Number of Workers: 10
Idle Timeout: 30
Session ID: 043f087a-da8b-4ae1-83fb-51c66eade912
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--conf spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin
--extra-py-files s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip
--extra-jars s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar,s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar
Waiting for session 043f087a-da8b-4ae1-83fb-51c66eade912 to get into ready status...
Session 043f087a-da8b-4ae1-83fb-51c66eade912 has been created.
Spark Extensions: com.esri.geoanalytics.s

In [2]:
import geoanalytics
from geoanalytics.sql import functions as ST
# Authenticate with the license file
geoanalytics.auth(username = "PTL_GAE",password = "MICXA_1008-10!")

In [ ]:
# Consolidated ptl_marketuniverse SQL: every base table, join, and non-spatial filter/derived
# column lives here in one place. All tables use the glue_catalog prefix consistently, and the
# result is cached + registered as v_mu so every downstream per-state spatial join (process_state)
# reuses this single materialization instead of re-running the full join chain per state.
combined_market_universe_df = spark.sql("""
    WITH sales_force_agg AS (
        SELECT
            duns_number,
            concat_ws(', ', collect_set(account_id)) AS sf_account_ids,
            concat_ws(', ', collect_set(CAST(confidence_code AS STRING))) AS sf_confidence_codes
        FROM ptl_marketuniverse.mu_salesforce_master
        GROUP BY duns_number
    )
    SELECT
        A.duns_number,
        A.business_name,
        A.parent_duns_number,
        A.headquarter_duns_number,
        A.dot_linkage,
        A.global_ultimate_duns_number,
        A.global_ultimate_indicator,
        A.global_ultimate_business_name,
        A.out_of_business_indicator,
        A.duns_linkage,
        A.number_of_family_members,
        A.penske_category,
        A.tradestyle_name,
        A.line_of_business,
        A.formatted_dot_linkage,
        A.us_1987_sic_1,
        A.naics,
        A.street_address,
        A.city_name,
        A.state_province_abbr,
        A.postal_code,
        A.county_name,
        A.latitude,
        A.longitude,
        A.employees_total,
        A.employees_here,
        A.sales_volume_us_dollars,
        A.telephone_number,
        A.chief_exec_officer_full_name,
        A.chief_exec_officer_title,
        A.first_executive_first_name,
        A.first_executive_last_name,
        A.first_executive_title,
        CASE
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 50000000 THEN '1. Mega ($50M+)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 10000000 THEN '2. Large ($10M - $50M)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 5000000 THEN '3. Upper Mid ($5M - $10M)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 2500000 THEN '4. Lower Mid ($2.5M - $5M)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 1000000 THEN '5. Small ($1M - $2.5M)'
            WHEN COALESCE(A.sales_volume_us_dollars, 0) >= 250000 THEN '6. Micro ($250K - $1M)'
            WHEN A.sales_volume_us_dollars IS NULL THEN '8. Unknown (No Data)'
            ELSE '7. Nano / Pre-Rev (<$250K)'
        END AS sales_target_segment,
        CASE
            WHEN A.duns_linkage LIKE '%|%' OR A.dot_linkage LIKE '%.%' THEN 'Yes'
            ELSE 'No'
        END AS is_child,
        P.confidence_code AS polk_confidence_code,
        P.total_fleet_size_gvw_3_8,
        R.confidence_code AS rigdig_confidence_code,
        R.ent_usdot_total_pwr,
        R.eqt_class_3to8_units,
        R.eqt_class_all_units,
        M.final_duns_number,
        SF.sf_account_ids,
        SF.sf_confidence_codes
    FROM ptl_marketuniverse.mu_dnb_data_master A
    LEFT JOIN glue_catalog.ptl_marketuniverse.mu_polk P
        ON A.duns_number = P.duns_number
    LEFT JOIN glue_catalog.ptl_marketuniverse.mu_rigdig R
        ON A.duns_number = R.duns_number
    LEFT JOIN ptl_marketuniverse.marketing_analytics M
        ON A.duns_number = M.site_duns_number
    LEFT JOIN sales_force_agg SF
        ON A.duns_number = SF.duns_number
    WHERE A.latitude IS NOT NULL
      AND A.longitude IS NOT NULL
""").cache()

print(f"Combined market universe row count: {combined_market_universe_df.count():,}")
combined_market_universe_df.show(10, truncate=False)
combined_market_universe_df.printSchema()

+-----------+--------------------+------------------+-----------------------+--------------+---------------------------+-------------------------+-----------------------------+-------------------------+--------------------+------------------------+---------------+--------------------+--------------------+---------------------+-------------+------+--------------------+---------------+-------------------+-----------+-------------+----------+-----------+---------------+--------------+-----------------------+----------------+----------------------------+------------------------+--------------------------+-------------------------+---------------------+--------------------+--------------------+------------------------+----------------------+-------------------+--------------------+-------------------+-----------------+------------------+-------------------+
|duns_number|       business_name|parent_duns_number|headquarter_duns_number|   dot_linkage|global_ultimate_duns_number|global_ultimate

In [ ]:
# Spatial: base SQL already filtered out null lat/long, so just build the point geometry.
df_dnb_pts = combined_market_universe_df.withColumn(
    "geometry",
    ST.point("longitude", "latitude", 4326)
)

# 2. Read GeoJSON boundary from S3
s3_geojson_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc_geopqt/DC_district_Bndy.geojson"

df_boundary = (
    spark.read.format("geojson")
    .load(s3_geojson_path)
    .filter(F.col("district") == "0860 WASHINGTON DC")
    .cache()
)

# 3. Perform Spatial Join using ST.contains (broadcast the single boundary row -- the small side)
spatial_start = time.time()
df_dnb_dc = df_dnb_pts.join(
    F.broadcast(df_boundary),
    ST.contains(df_boundary["geometry"], df_dnb_pts["geometry"]),
    how="inner"
).select(df_dnb_pts["*"]).cache()

dc_count = df_dnb_dc.count()
spatial_elapsed = time.time() - spatial_start

# 4. Preview DC Filtered Market Universe
df_dnb_dc.select("duns_number", "business_name", "latitude", "longitude").show(5, truncate=False)
print(f"Total D&B Master Records in District 0860 WASHINGTON DC: {dc_count:,} (spatial join took {spatial_elapsed:,.1f}s)")

In [8]:
df_boundary.count()

1


## Esri Geocoding Patch (v3) — manual ArcGIS Pro round-trip

D&B's own lat/long is sometimes wrong (bad interpolation, stale geocode, etc.). Since this Glue environment has no outbound network path to call Esri's geocoding service directly, this is a **manual round-trip** instead:

1. **Export** (next cell) — write the territory-scoped D&B records (DC/MD/VA, inside `df_boundary`) to a CSV on S3, with `duns_number` as the stable join key plus the address fields needed for geocoding.
2. **Manual step (outside this notebook)** — download that CSV, run it through ArcGIS Pro's Geocode Addresses tool against an Esri locator, then upload the result (same `duns_number` key, plus Esri's match score and X/Y columns) to the `geocoded_result_s3_path` below.
3. **Patch back in** (further down) — only Esri matches scoring **above the threshold** replace the original D&B coordinates ("patch match"); everything else keeps the original D&B lat/long. Either way, every record gets a `geo_locator_match` Yes/No field so it's always visible which coordinate source was actually used.

Re-run from the patch cell onward (not the export cell) once you've uploaded the geocoded file — re-running the export would just regenerate the same input file, harmless but pointless.

In [ ]:
# CONFIG for the geocoding round-trip. Column names on the geocoded side match
# ArcGIS Pro's default Geocode Addresses output field names (Score, X, Y) - adjust
# here if your export settings produced different column names.
GEOCODE_CONFIG = {
    "export_csv_s3_path": "s3://pske-prd-customerexperienceadhoc/spatial_analysis/geocode_roundtrip/dnb_territory_export_for_arcgis_pro",
    "geocoded_result_s3_path": "s3://pske-prd-customerexperienceadhoc/spatial_analysis/geocode_roundtrip/dnb_territory_geocoded_from_arcgis_pro.csv",
    "join_key": "duns_number",
    "esri_score_col": "Score",
    "esri_lat_col": "Y",
    "esri_lon_col": "X",
    "score_threshold": 90,
}

In [ ]:
# STEP 1 - EXPORT: territory-scoped (DC/MD/VA, inside df_boundary) D&B addresses,
# for manual geocoding in ArcGIS Pro. duns_number is carried through as the join key
# so the geocoded result can be matched back to combined_market_universe_df later.
territory_dnb_pts = (
    combined_market_universe_df
    .filter(F.col("state_province_abbr").isin("DC", "MD", "VA"))
    .withColumn("pointgeom", ST.point("longitude", "latitude", sr=4326))
    .st.set_geometry_field("pointgeom")
)

territory_dnb_in_boundary = (
    territory_dnb_pts
    .join(
        F.broadcast(df_boundary),
        ST.contains(df_boundary["geometry"], territory_dnb_pts["pointgeom"]),
        how="inner",
    )
    .select(territory_dnb_pts["*"])
)

# Field names below match Esri's standard composite locator address fields
# (street_address/city/county/state/zip) rather than the raw D&B column names,
# so ArcGIS Pro's Geocode Addresses tool can auto-match them without manual
# field mapping. zip is truncated to 5 digits since D&B's postal_code can carry
# ZIP+4.
geocode_export_df = territory_dnb_in_boundary.select(
    "duns_number",
    "business_name",
    F.col("street_address").alias("street_address"),
    F.col("city_name").alias("city"),
    F.col("county_name").alias("county"),
    F.col("state_province_abbr").alias("state"),
    F.substring(F.col("postal_code"), 1, 5).alias("zip"),
    "latitude",
    "longitude",
)

export_row_count = geocode_export_df.count()
geocode_export_df.coalesce(1).write.mode("overwrite") \
    .option("header", "true").option("quoteAll", "true") \
    .csv(GEOCODE_CONFIG["export_csv_s3_path"])

print(f"Exported {export_row_count:,} territory D&B records to {GEOCODE_CONFIG['export_csv_s3_path']}")
print("Next: download this CSV, geocode it in ArcGIS Pro, then upload the result "
      f"(same duns_number key, plus Score/X/Y) to {GEOCODE_CONFIG['geocoded_result_s3_path']}")

### STOP — manual step required here

1. Download the CSV(s) written to `GEOCODE_CONFIG["export_csv_s3_path"]`.
2. In ArcGIS Pro, run **Geocode Addresses** against your Esri locator using `street_address`/`city`/`county`/`state`/`zip` — these are named to match Esri's standard composite locator fields, so field mapping should auto-detect.
3. Export the geocoded result's attribute table (including `duns_number`, `Score`, `X`, `Y`) as a CSV.
4. Upload it to `GEOCODE_CONFIG["geocoded_result_s3_path"]`.

Only continue to the next cell once that file exists at that path.

In [ ]:
# STEP 2 - PATCH BACK: only Esri matches scoring ABOVE the threshold replace the
# original D&B lat/long ("patch match"); everything else keeps the original
# coordinates. geo_locator_match records which source actually won, for every row -
# including D&B records that were never in the geocode export at all (e.g. outside
# DC/MD/VA), which correctly get geo_locator_match = "No" via the left join below.
esri_geocoded_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(GEOCODE_CONFIG["geocoded_result_s3_path"])
    .select(
        F.col(GEOCODE_CONFIG["join_key"]).alias("duns_number"),
        F.col(GEOCODE_CONFIG["esri_lat_col"]).cast("double").alias("esri_latitude"),
        F.col(GEOCODE_CONFIG["esri_lon_col"]).cast("double").alias("esri_longitude"),
        F.col(GEOCODE_CONFIG["esri_score_col"]).cast("double").alias("esri_score"),
    )
)

is_high_confidence_match = (
    F.col("esri_score").isNotNull() & (F.col("esri_score") > GEOCODE_CONFIG["score_threshold"])
)

combined_market_universe_df = (
    combined_market_universe_df
    .join(esri_geocoded_df, on="duns_number", how="left")
    .withColumn("geo_locator_match", F.when(is_high_confidence_match, "Yes").otherwise("No"))
    .withColumn(
        "latitude",
        F.when(F.col("geo_locator_match") == "Yes", F.col("esri_latitude")).otherwise(F.col("latitude")),
    )
    .withColumn(
        "longitude",
        F.when(F.col("geo_locator_match") == "Yes", F.col("esri_longitude")).otherwise(F.col("longitude")),
    )
    .drop("esri_latitude", "esri_longitude", "esri_score")
    .cache()
)

print(f"Patched D&B universe row count: {combined_market_universe_df.count():,}")
combined_market_universe_df.groupBy("geo_locator_match").count().show()

In [35]:
print("\nRefined Breakdown by Sales Target Segment:")
(
    df_dnb_dc
    .groupBy("sales_target_segment")
    .count()
    .orderBy("sales_target_segment")
    .show(truncate=False)
)


Refined Breakdown by Sales Target Segment:
+--------------------------+------+
|sales_target_segment      |count |
+--------------------------+------+
|1. Mega ($50M+)           |1203  |
|2. Large ($10M - $50M)    |2943  |
|3. Upper Mid ($5M - $10M) |3619  |
|4. Lower Mid ($2.5M - $5M)|4309  |
|5. Small ($1M - $2.5M)    |7714  |
|6. Micro ($250K - $1M)    |22109 |
|7. Nano / Pre-Rev (<$250K)|125519|
+--------------------------+------+


In [ ]:
combined_market_universe_df.groupBy("state_province_abbr").count().show(truncate=False)

+-------------------+-----+
|state_province_abbr|count|
+-------------------+-----+
|NULL               |1    |
|VA                 |73580|
|MD                 |71576|
|DC                 |22184|
|WA                 |1    |
|FL                 |16   |
|NY                 |5    |
|SC                 |3    |
|TX                 |6    |
|IL                 |4    |
|NC                 |3    |
|MA                 |2    |
|CA                 |8    |
|MN                 |2    |
|ME                 |2    |
|CO                 |2    |
|AL                 |2    |
|DE                 |3    |
|MS                 |1    |
|NJ                 |1    |
+-------------------+-----+
only showing top 20 rows


In [44]:
# Configuration & Paths
S3_TEMP_BASE = "s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/temp_stage_output"
BASE_S3_PATH = "s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/State_Level_US_Parcels"


In [ ]:
def process_state(state_code, folder=None, state_timings=None):
    """Run D&B + parcel spatial join for a state. Every D&B record is enriched with
    its best-matching parcel (ANY zoning/land use, not pre-filtered to non-residential),
    plus a business-facing parcel_nonresidential_match Yes/No field computed after the
    join. Pre-filtering parcels before the join (the old approach) silently drops
    zoning/LBCS/owner data for any D&B record that happens to sit on a residential
    parcel, since that parcel never enters the join candidate set at all.

    D&B records are scoped to the actual `df_boundary` territory polygon (defined
    earlier - it spans parts of DC, MD, AND VA, since it's a Penske service
    territory, not a state line), not just the state_province_abbr label. Running
    ST.contains against the full D&B universe in one shot didn't finish, so the
    state filter narrows the candidate rows first (cheap attribute filter) and the
    boundary check runs on that much smaller per-state subset - the same
    broadcast-boundary pattern already proven fast for DC alone earlier in this
    notebook.

    PERFORMANCE: the D&B side is already boundary-scoped (small), but the parcel
    side was still scanning every parcel statewide - for MD that's ~2.49M parcels
    even though the territory only covers a small DC-adjacent slice of the state.
    A bounding-box pre-filter (derived from the boundary-scoped D&B points' own
    lat/lon extent, buffered so it's a strict superset of the true boundary) cuts
    the parcel candidate set down to a cheap rectangle-intersection test before the
    expensive exact ST.contains runs, instead of testing every statewide parcel
    against every D&B point.

    Two fields ship in the final output, answering two different questions:
    - parcel_match (Y/N): was there any parcel match at all?
    - parcel_nonresidential_match (Yes/No): of that match (if any), is it non-residential?
    parcel_match = "Y" + parcel_nonresidential_match = "No" is a valid, expected
    combination (matched a residential parcel) - not a contradiction."""
    folder = folder or state_code
    start_time = time.time()
    print(f"=== STARTING {state_code} PROCESSING ===")

    # 1. Read ALL State Parcels - no non-residential pre-filter. De-dup on parcel id
    #    (Regrid can carry duplicate rows per parcel) before the spatial join;
    #    dropDuplicates is far cheaper here than a Window/row_number pass.
    s3_path = f"{BASE_S3_PATH}/{folder}/"
    df_state = (
        spark.read.parquet(s3_path)
        .withColumn("source_state", F.lit(state_code))
        .dropDuplicates(["parcelnu_1"])
    )

    # 2. Parcel geometry + fields to keep: parcel id/owner/use fields plus all four
    #    LBCS dimensions (Activity, Function, Structure, Site) each paired with its
    #    human-readable description, so the business can interpret land use directly
    #    across multiple attributes, not just the single Activity code used for the
    #    residential/non-residential determination below.
    parcel_geom = df_state.withColumn(
        "polygeom", ST.geom_from_binary("geometry", sr=4326)
    ).select(
        "polygeom", "parcelnu_1", "usecode", "zoning", "zoning_des",
        "zoning_typ", "zoning_sub",
        "lbcs_activ", "lbcs_act_1",
        "lbcs_funct", "lbcs_fun_1",
        "lbcs_strucs", "lbcs_str_1",
        "lbcs_site", "lbcs_site_",
        "lbcs_owner", "lbcs_own_1",
        "housing_af",
        "owner", "usedesc",
    ).st.set_geometry_field("polygeom")

    # 3. Filter D&B Universe for Target State (cheap pre-filter to shrink the
    #    candidate set), THEN scope to the actual df_boundary territory polygon -
    #    this is the real geographic filter; state_province_abbr alone is just a
    #    mailing-address label and can disagree with the true boundary at the edges.
    dnb_state_pts = (
        combined_market_universe_df
        .filter(F.col("state_province_abbr") == state_code)
        .withColumn("pointgeom", ST.point("longitude", "latitude", sr=4326))
        .st.set_geometry_field("pointgeom")
    )

    dnb_df_geom = (
        dnb_state_pts
        .join(
            F.broadcast(df_boundary),
            ST.contains(df_boundary["geometry"], dnb_state_pts["pointgeom"]),
            how="inner",
        )
        .select(dnb_state_pts["*"])
        .st.set_geometry_field("pointgeom")
        .cache()
    )

    t_diag = time.time()
    dnb_boundary_count = dnb_df_geom.count()
    print(f"    {state_code}: D&B records inside boundary: {dnb_boundary_count:,} "
          f"({time.time() - t_diag:.1f}s to count)")

    # 3b. Bounding-box pre-filter on the parcel side, derived from the D&B side's
    #     own lat/lon extent (buffered ~0.02 deg / ~2km so no true match near the
    #     edge gets clipped). This is a coarse rectangle-intersection test, much
    #     cheaper than the exact polygon ST.contains that follows, so it should
    #     shrink e.g. MD's 2.49M statewide parcels down to just the ones near the
    #     territory before the expensive exact check runs.
    t_diag = time.time()
    bbox_stats = dnb_df_geom.select(
        F.min("longitude").alias("min_lon"), F.max("longitude").alias("max_lon"),
        F.min("latitude").alias("min_lat"), F.max("latitude").alias("max_lat"),
    ).first()

    buffer_deg = 0.02
    min_lon, max_lon = bbox_stats["min_lon"] - buffer_deg, bbox_stats["max_lon"] + buffer_deg
    min_lat, max_lat = bbox_stats["min_lat"] - buffer_deg, bbox_stats["max_lat"] + buffer_deg
    bbox_wkt = (
        f"POLYGON(({min_lon} {min_lat}, {max_lon} {min_lat}, "
        f"{max_lon} {max_lat}, {min_lon} {max_lat}, {min_lon} {min_lat}))"
    )
    bbox_df = (
        spark.createDataFrame([(bbox_wkt,)], ["wkt"])
        .withColumn("bboxgeom", ST.geom_from_wkt("wkt", sr=4326))
        .st.set_geometry_field("bboxgeom")
    )

    parcel_geom_prefiltered = (
        parcel_geom
        .join(
            F.broadcast(bbox_df),
            ST.intersects(parcel_geom["polygeom"], bbox_df["bboxgeom"]),
            how="inner",
        )
        .select(parcel_geom["*"])
    )

    prefiltered_count = parcel_geom_prefiltered.count()
    print(f"    {state_code}: parcels after bbox pre-filter: {prefiltered_count:,} "
          f"({time.time() - t_diag:.1f}s)")

    dnb_side = F.broadcast(dnb_df_geom)

    # 4. Spatial Join (ST.contains) against the bbox-prefiltered parcels. Right join
    #    so every D&B record survives even with zero parcel match (parcel_match
    #    below makes that case explicit rather than leaving it indistinguishable
    #    from a residential match).
    dnb_parcel_intersect = parcel_geom_prefiltered.join(
        dnb_side,
        ST.contains(parcel_geom_prefiltered["polygeom"], dnb_side["pointgeom"]),
        how="right"
    )

    # 5. Non-residential determination, computed AFTER the join against whatever
    #    parcel actually matched. Null-safe by construction: has_parcel gates the
    #    whole expression, so a record with no match (or null zoning/LBCS on a real
    #    match) never silently reads as "Yes" the way `lbcs_activ IS NULL` alone did
    #    in the old pre-filter condition.
    has_parcel = F.col("parcelnu_1").isNotNull()
    zone_nonresidential = F.col("zoning_typ").isNotNull() & (F.col("zoning_typ") != "Residential")
    lbcs_confirmed_residential = F.col("lbcs_activ").isNotNull() & F.col("lbcs_activ").cast("int").between(1000, 1999)
    lbcs_household_text = F.col("lbcs_act_1").isNotNull() & F.col("lbcs_act_1").ilike("%Household%")
    lbcs_nonresidential = ~lbcs_confirmed_residential & ~lbcs_household_text
    is_nonresidential_match = has_parcel & (zone_nonresidential | lbcs_nonresidential)

    dnb_parcel_intersect = (
        dnb_parcel_intersect
        .withColumn("parcel_match", F.when(has_parcel, "Y").otherwise("N"))
        .withColumn("parcel_nonresidential_match", F.when(is_nonresidential_match, "Yes").otherwise("No"))
        # Tie-break priority when a D&B point overlaps multiple parcels: prefer any
        # match over none, then prefer a non-residential match, then lowest lbcs_activ.
        .withColumn("_nonres_priority", F.when(F.col("parcel_nonresidential_match") == "Yes", 0).otherwise(1))
    )

    # 6. Window Deduplication - one best parcel match per D&B company.
    match_dedup_window = Window.partitionBy("duns_number").orderBy(
        F.col("parcel_match").desc(),
        F.col("_nonres_priority").asc(),
        F.col("lbcs_activ").asc_nulls_last()
    )

    no_dup_df = (
        dnb_parcel_intersect
        .withColumn("rn", F.row_number().over(match_dedup_window))
        .filter(F.col("rn") == 1)
        .drop("rn", "_nonres_priority")
    )

    # Single write/action triggers the pipeline safely in a single DAG execution
    print(f"=== COMPLETED {state_code} IN {time.time() - start_time:,.1f}s ===")

    # Preserve spatial point geometry as shape column
    no_dup_df = no_dup_df.drop("polygeom").withColumnRenamed("pointgeom", "shape")
    no_dup_df = no_dup_df.st.set_geometry_field("shape")

    # 7. MATERIALIZATION STEP: Write to Parquet Once to Break Lineage
    parquet_path = f"{S3_TEMP_BASE}/parquet/dedup_{state_code}/"
    no_dup_df.write.mode("overwrite").parquet(parquet_path)

    # 8. Read back materialized data for lightweight downstream exports
    staged_df = spark.read.parquet(parquet_path).st.set_geometry_field("shape")

    # CSV Export (single file per district)
    csv_path = f"{S3_TEMP_BASE}/csv/dedup_{state_code}/"
    staged_df.drop("shape").drop("geometry").coalesce(1).write.mode("overwrite") \
        .option("header", "true").option("quoteAll", "true").csv(csv_path)

    # GeoParquet Export (single file per district)
    geoparquet_path = f"{S3_TEMP_BASE}/geoparquet/dedup_{state_code}/"
    staged_df.drop("geometry").coalesce(1).write.format("geoparquet").mode("overwrite").save(geoparquet_path)

    record_count = staged_df.count()
    elapsed = time.time() - start_time
    if state_timings is not None:
        state_timings[state_code] = elapsed
    print(f"=== {state_code} COMPLETED: {record_count:,} records written in {elapsed:,.1f}s (Parquet, CSV, & GeoParquet) ===\n")
    return staged_df

In [ ]:
# Runs all three per-state spatial joins through the shared, optimized process_state() pipeline.
# Edit this list to re-run only the state(s) that failed, e.g. STATES_TO_RUN = ["VA"]
STATES_TO_RUN = ["DC", "MD", "VA"]
state_final_dfs = {}
state_timings = {}
for state_code in STATES_TO_RUN:
    state_final_dfs[state_code] = process_state(state_code, state_timings=state_timings)
print(f"Completed states: {list(state_final_dfs.keys())}")

In [ ]:
print(STATES_TO_RUN)

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

# Map state abbreviations to S3 folder names
states_map = {
    "DC": "DC",
    "MD": "MD", # Update folder name if MD folder is named 'Maryland'
    "VA": "VA"  # Update folder name if VA folder is named 'Virginia'
}

base_s3_path = "s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/State_Level_US_Parcels"

state_all_dfs = []
state_non_res_dfs = []

# Define standard Non-Residential filter condition
# Non-residential zoning OR Non-residential land use (LBCS)
non_res_condition = (
    (F.col("zoning_typ") != "Residential") |
    F.col("lbcs_activ").isNull() |
    ~(
        (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
        (F.col("lbcs_act_1").ilike("%Household%"))
    )
)

for state_code, folder in states_map.items():
    s3_path = f"{base_s3_path}/{folder}/"

    # Cache once -- this DataFrame is scanned by every count below, so caching avoids
    # re-reading the parquet source from S3 for each metric.
    df_state = spark.read.parquet(s3_path).withColumn("source_state", F.lit(state_code)).cache()
    df_state_non_res = df_state.filter(non_res_condition)

    # Single aggregation pass instead of four separate .count() actions per state.
    stats = df_state.select(
        F.count(F.lit(1)).alias("total_cnt"),
        F.sum(F.when(F.col("zoning_typ") != "Residential", 1).otherwise(0)).alias("non_res_zoning_cnt"),
        F.sum(F.when(
            F.col("lbcs_activ").isNull() |
            ~(
                (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
                (F.col("lbcs_act_1").ilike("%Household%"))
            ), 1).otherwise(0)).alias("non_res_landuse_cnt"),
        F.sum(F.when(non_res_condition, 1).otherwise(0)).alias("non_res_cnt")
    ).first()

    print(f"--- {state_code} Parcel Summary ---")
    print(f"  Total {state_code} counts: {stats['total_cnt']:,}")
    print(f"  Non-Residential Zoning counts: {stats['non_res_zoning_cnt']:,}")
    print(f"  Non-Residential Landuse counts: {stats['non_res_landuse_cnt']:,}")
    print(f"  Combined Non-Residential {state_code} counts: {stats['non_res_cnt']:,}\n")

    state_all_dfs.append(df_state)
    state_non_res_dfs.append(df_state_non_res)

# 1. Combine ALL state DataFrames into a single unified DataFrame
parcels_all_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), state_all_dfs)

# 2. Combine NON-RESIDENTIAL state DataFrames into a single unified DataFrame
non_res_parcels_all_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), state_non_res_dfs)

print(f"Combined Tri-State All Parcels Count: {parcels_all_df.count():,}")
print(f"Combined Tri-State Non-Residential Parcels Count: {non_res_parcels_all_df.count():,}")

--- DC Parcel Summary ---
  Total DC counts: 207,339
  Non-Residential Zoning counts: 34,068
  Non-Residential Landuse counts: 34,361
  Combined Non-Residential DC counts: 54,706

--- MD Parcel Summary ---
  Total MD counts: 2,489,839
  Non-Residential Zoning counts: 713,696
  Non-Residential Landuse counts: 336,166
  Combined Non-Residential MD counts: 821,094

--- VA Parcel Summary ---
  Total VA counts: 4,240,836
  Non-Residential Zoning counts: 2,321,338
  Non-Residential Landuse counts: 1,051,094
  Combined Non-Residential VA counts: 2,509,716

Combined Tri-State All Parcels Count: 6,938,014
Combined Tri-State Non-Residential Parcels Count: 3,385,516


In [21]:
parcel_nonres_geom = non_res_parcels_all_df.withColumn("polygeom", ST.geom_from_binary("geometry", sr=4326))
parcel_nonres_geom = parcel_nonres_geom.select("polygeom","parcelnu_1","usecode","zoning","zoning_des","zoning_typ","zoning_sub","lbcs_activ","owner","usedesc","zoning_des")
print(parcel_nonres_geom.count())
parcel_nonres_geom.show(5)
parcel_nonres_geom.printSchema()

2435018
+--------------------+--------------+-------+------+--------------------+----------+----------+----------+--------------------+-----------+--------------------+
|            polygeom|    parcelnu_1|usecode|zoning|          zoning_des|zoning_typ|zoning_sub|lbcs_activ|               owner|    usedesc|          zoning_des|
+--------------------+--------------+-------+------+--------------------+----------+----------+----------+--------------------+-----------+--------------------+
|{"rings":[[[-76.9...|              |       | PDR-1|Production Distri...|   Special|   Special|      NULL|                    |           |Production Distri...|
|{"rings":[[[-76.9...|0000Unassessed|       |    UZ|             Unzoned|   Special|   Special|      NULL|                    |           |             Unzoned|
|{"rings":[[[-77.0...|      00010843|    191|    UZ|             Unzoned|   Special|   Special|      NULL|UNITED STATES OF ...|Vacant-True|             Unzoned|
|{"rings":[[[-77.0...|    

In [19]:
geoparquet_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/dnb_parcel_match_dc"
dc_pqt = spark.read.parquet(geoparquet_s3_path)
print(dc_pqt.count())

AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3://pske-prd-customerexperienceadhoc/spatial_analysis/dnb_parcel_match_dc.


In [ ]:
from datetime import datetime

print("=" * 78)
print("D&B NON-RESIDENTIAL PROSPECT MATCH -- BUSINESS DELIVERY SUMMARY")
print("=" * 78)
print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Territories requested: {', '.join(STATES_TO_RUN)}")
print()
print("PROCESS COMPLETED")
print("1. Loaded D&B company records with valid latitude and longitude coordinates.")
print("2. Joined D&B records with Polk, RigDig, marketing analytics, and Sales Force data.")
print("3. Added sales target segments and child-company classification.")
print("4. Patched high-confidence Esri geocode results into D&B coordinates (geo_locator_match).")
print("5. Matched company locations to parcel boundaries using spatial point-in-polygon analysis.")
print("6. Selected one best parcel match per D&B company where multiple parcels overlapped.")
print("7. Wrote territory-level CSV, Parquet, and GeoParquet delivery files.")
print()
print("QUALITY AND PERFORMANCE CHECKS")
print("- Records without usable coordinates were excluded from spatial matching.")
print("- Spatial joins were run separately by territory to control workload and memory use.")
print("- Parcel and company data were filtered before spatial matching.")
print("- Spatial join row counts and elapsed times were captured for each territory.")
print("- Output row counts were checked after each territory completed.")
print()
print("DELIVERY RESULTS")
total_records = 0
total_geocoded = 0
total_nonres = 0
completed_states = []
for state_code in STATES_TO_RUN:
    staged_df = state_final_dfs.get(state_code)
    if staged_df is None:
        print(f"{state_code}: NOT COMPLETED")
        continue
    record_count = staged_df.count()
    geocoded_count = staged_df.filter(F.col("geo_locator_match") == "Yes").count()
    nonres_count = staged_df.filter(F.col("parcel_nonresidential_match") == "Yes").count()
    total_records += record_count
    total_geocoded += geocoded_count
    total_nonres += nonres_count
    completed_states.append(state_code)
    runtime_seconds = state_timings.get(state_code)
    runtime_text = f"{runtime_seconds:,.1f} seconds" if runtime_seconds is not None else "not recorded"
    print(f"{state_code}: {record_count:,} matched non-residential prospect records")
    print(f"    Runtime:               {runtime_text}")
    print(f"    Esri-geocoded (patch): {geocoded_count:,} of {record_count:,} "
          f"({100 * geocoded_count / record_count:.1f}%)")
    print(f"    Non-residential match: {nonres_count:,} of {record_count:,} "
          f"({100 * nonres_count / record_count:.1f}%)")
    print(f"    CSV:        {S3_TEMP_BASE}/csv/dedup_{state_code}/")
    print(f"    GeoParquet: {S3_TEMP_BASE}/geoparquet/dedup_{state_code}/")
print()
print(f"Completed territories: {', '.join(completed_states) if completed_states else 'None'}")
print(f"Total matched records:        {total_records:,}")
print(f"Total Esri-geocoded (patch):  {total_geocoded:,}")
print(f"Total non-residential match: {total_nonres:,}")
print()
print("DELIVERY NOTE")
print("The completed territory files above are ready for handoff to the data delivery team.")
print("Each record represents a D&B company matched to the best available parcel result,")
print("with geo_locator_match showing whether Esri's geocode replaced the original D&B")
print("coordinate, and parcel_nonresidential_match showing the matched parcel's zoning/LBCS.")
print("Runtime includes parcel preparation, spatial matching, deduplication, and file delivery for each territory.")
print("=" * 78)